In [2]:
!pip install transformers fugashi ipadic unidic-lite

In [21]:
MODEL_PATH = "line-corporation/line-distilbert-base-japanese"
DATASET_NAME = "kanhatakeyama/japanese-corpus-categorized"

In [22]:
import re 
import json
import unicodedata
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer

#設定
sentences_file_name = "sentences_all.jsonl"
MAX_ROWS = 5000000
BATCH_SIZE = 5000
LABEL_0_COUNT = 1000000
LABEL_1_COUNT = 1000000
MIN_PAREN_CHARS = 3  # 括弧内の最小文字数


tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True)

# 有効な文字のみからなる文のパターン
pattern = re.compile(r'^[（）()\u3040-\u309F\u30A0-\u30FF\u4E00-\u9FFF、a-zA-Z0-9]+$')
#句読点が入る括弧は除く
incorrect_pattern = re.compile(r'\([^)]*?。[^\)]*?\)')
#丸括弧が連続する文を除外
consecutive_pattern = re.compile(r'\)\s*\(')
#文章全体が括弧で囲まれている場合を除外
whole_sentence_pattern = re.compile(r'^\(.*\)。?$')

def is_valid_parentheses_text(text):
    """括弧の対応が正しいかを確認する"""
    stack = []
    for i, t in enumerate(text):
        if t in ("(","（"):
            if stack: return False
            stack.append(i)
            
        elif t in (")","）"):
            if not stack: return False
            j = stack.pop()
            if i - j <= 1: return False
    return not stack


def has_enough_chars_in_parens(text, min_chars=MIN_PAREN_CHARS):
    """全ての括弧内の文字数がmin_chars以上かを確認する"""
    matches = list(re.finditer(r'\(([^)]+)\)', text))
    if not matches:
        return False
    return all(len(m.group(1).strip()) >= min_chars for m in matches)


def process_batch(batch, tokenizer, fout, label_counter):
    """バッチ単位でフィルタリングしてjsonlに書き出す"""
    if not batch:
        return
    
    #正規化と句点の付与
    normalized = []
    for s in batch:
        s_norm = unicodedata.normalize("NFKC", s).strip()
        if not s_norm: continue
        if not s_norm.endswith("。"):
            s_norm += "。"
        if pattern.fullmatch(s_norm.rstrip("。")):
            normalized.append(s_norm)
    if not normalized:
        return
    
    filtered_for_tokenize = []
    for s in normalized:
        has_left = ("(" in s or "（" in s)
        has_right = (")" in s or "）" in s)
        
        # 括弧がない かつ すでに上限なら、スキップする
        if not has_left and not has_right:
            if label_counter[0] >= LABEL_0_COUNT:
                continue
        
        filtered_for_tokenize.append(s)
    
    if not filtered_for_tokenize:
        return 
    
    encode = tokenizer(filtered_for_tokenize, truncation=False, padding=False)
    
    for s, ids in zip(filtered_for_tokenize, encode["input_ids"]):
        # 短すぎる・長すぎる文を除外
        if not(10 <= len(ids)<=128):
            continue
            
        has_left = ("(" in s or "（" in s)
        has_right = (")" in s or "）" in s)
        
        label = 0
        if has_left and has_right:
             # 括弧の対応・句点またぎ・文字数の条件をすべて満たす場合のみlabel=1
            if not incorrect_pattern.search(s) and is_valid_parentheses_text(s):
                if not has_enough_chars_in_parens(s,min_chars=MIN_PAREN_CHARS):
                    continue
                if consecutive_pattern.search(s):
                    continue
                if whole_sentence_pattern.search(s):
                    continue
                label = 1
            else:
                continue
        elif has_left or has_right:
            # 片方だけの括弧は除外
            continue
            
        if label == 0 and label_counter[0] >= LABEL_0_COUNT:
            continue
            
        json.dump({"text":s, "label": label}, fout, ensure_ascii=False)
        fout.write("\n")
        label_counter[label] += 1
        

#メイン処理
dataset = load_dataset(DATASET_NAME, split="train", streaming=True)
label_counter = {0:0, 1:0}
current_batch = []

with open(sentences_file_name, "w", encoding="utf-8") as fout:
    pbar = tqdm(total=MAX_ROWS)
    
    for i, article in enumerate(dataset):
        if i >= MAX_ROWS:
            break
        if label_counter[1] >= LABEL_1_COUNT:
            print(f"\n目標の括弧あり文章（{LABEL_1_COUNT}件）に達しました。")
            break
        
        
        text = article.get("text", "")
        if not text: continue
        
        # 記事を文単位に分割してバッチに追加
        for line in text.splitlines():
            for s in line.split("。"):
                s = s.strip()
                if 5 <= len(s) <= 200:
                    current_batch.append(s)
        
        if len(current_batch) >= BATCH_SIZE:
            process_batch(current_batch, tokenizer, fout, label_counter)
            pbar.set_postfix({
                "あり": f"{label_counter[1]}件", 
                "なし": f"{label_counter[0]}件"
            })
            current_batch = []
            pbar.update(i-pbar.n)
            
            
    
    # 残りのバッチを処理
    process_batch(current_batch, tokenizer, fout, label_counter)
    pbar.close()
    
            
print(f"\n完了！\n丸括弧あり: {label_counter[1]}\n丸括弧なし: {label_counter[0]}")


Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

  0%|          | 0/5000000 [00:00<?, ?it/s]


目標の括弧あり文章（1000000件）に達しました。

完了！
丸括弧あり: 1000076
丸括弧なし: 1000000


In [19]:
import json
from sklearn.model_selection import train_test_split
import random
from collections import Counter

data = []
with open("sentences_all.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data.append(json.loads(line))
    
data_no = [d for d in data if d["label"] == 0]
data_pa = [d for d in data if d["label"] == 1]

print(*data_pa[:100], sep='\n')

{'text': '冷蔵保存の場合はなるべく早めに(5日以内)、冷凍保存は1ヶ月以内ににお召し上がりください。', 'label': 1}
{'text': '鍋(実際は、適当な鍋がなかったのでフライパン)に入れ、水を入れ、沸騰させました。', 'label': 1}
{'text': '縦薄切りにする場合、縦半分に切って、切り口を下にして縦に置き、繊維に沿って、端から料理に応じた厚さに切ります(画像のもの)。', 'label': 1}
{'text': '今度は温経湯(ウンケイトウ)といって、やはり血の流れを改善する薬です。', 'label': 1}
{'text': '呉茱萸(ゴシュユ)という、先生曰く、くさーいのが入っているんですが、私には全然くさく感じないんです。', 'label': 1}
{'text': '1、鍋に湯を沸かし、少量の塩(分量外)を加えてその中に鶏肉を入れて弱火でゆっくり茹で火が通ったら取り出し、再び火を強め沸騰したらもやしを20秒加えてざるにあける。', 'label': 1}
{'text': '・オースは140キロパスカル(超高圧)です。', 'label': 1}
{'text': '(なんとなく親心)。', 'label': 1}
{'text': 'オートバックス(セコハン広場)のある場所はもとは駐車場のやたらに広い豚太郎のあった場所。', 'label': 1}
{'text': '王将(餃子のでは無く)と同レベル。', 'label': 1}
{'text': 'ギョーザの美味い中華そば屋(ラーメン屋)はどこか教えてください。', 'label': 1}
{'text': 'とん太(ふとし)ですね。', 'label': 1}
{'text': 'とん太(ふとし)はA級に不潔です。', 'label': 1}
{'text': 'とん太(ふとし)は腹を切って死ぬべき。', 'label': 1}
{'text': 'そこへ生クリームを加え、醤油、塩、黒胡椒で味を調える(濃度も好みで調整してください)。', 'label': 1}
{'text': '(KKモントレイユさん)。', 'label': 1}
{'text': '3に入れ味を見て、オリーブオイル、塩こしょう(分量外)で調味する。', 'label': 1}
